In [2]:
# Reusable business logic is maintained in src/analysis.py.
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.analysis import build_licensing_units, compare_units


from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------
# 1. Project setup
# --------------------------------------------------

PROJECT_ROOT = Path.cwd()

# If the notebook is being run from the notebooks folder,
# move one level up to the project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", ANALYSIS_DIR)

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Load and prepare movie data
# --------------------------------------------------

ANALYSIS_DIR = PROJECT_ROOT / "data" / "processed"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

movies = pd.read_csv(
    ANALYSIS_DIR / "movies.csv"
)

movies["release_year"] = pd.to_numeric(
    movies["release_year"], errors="coerce"
)

movies["budget"] = pd.to_numeric(
    movies["budget"], errors="coerce"
)

movies["revenue"] = pd.to_numeric(
    movies["revenue"], errors="coerce"
)

# Do not silently treat missing or invalid financial values as zero.
movies["profit"] = movies["revenue"] - movies["budget"]

movies["movie_roi_pct"] = np.where(
    movies["budget"] > 0,
    (movies["profit"] / movies["budget"]) * 100,
    np.nan
)

movies["revenue_multiple"] = np.where(
    movies["budget"] > 0,
    movies["revenue"] / movies["budget"],
    np.nan
)


# --------------------------------------------------
# 2. Classify licensing units within each period
# --------------------------------------------------



# --------------------------------------------------
# 3. Build the two analysis periods
# --------------------------------------------------

periods = {
    "All release years": movies.copy(),
    "2000 onward": movies[
        movies["release_year"] >= 2000
    ].copy()
}

unit_datasets = {}
sensitivity_datasets = {}

for period_name, period_movies in periods.items():

    # Primary analysis: exclude single-film collection candidates.
    units, classified = build_licensing_units(
        period_movies,
        include_single_collections=False
    )

    unit_datasets[period_name] = units

    # Sensitivity analysis: treat single-film collection candidates
    # as standalone licensing opportunities.
    sensitivity_units, _ = build_licensing_units(
        period_movies,
        include_single_collections=True
    )

    sensitivity_datasets[period_name] = sensitivity_units

    print(f"\n{period_name}")
    print(classified["licensing_type"].value_counts())


# --------------------------------------------------
# 4. Compare licensing performance
# --------------------------------------------------



primary_comparison = compare_units(
    unit_datasets,
    "Primary: multi-film bundles only"
)

sensitivity_comparison = compare_units(
    sensitivity_datasets,
    "Sensitivity: single-film collections treated as standalone"
)

comparison_df = pd.concat(
    [primary_comparison, sensitivity_comparison],
    ignore_index=True
)

print("\nLICENSING PERFORMANCE COMPARISON")
print(comparison_df.round(2).to_string(index=False))


# --------------------------------------------------
# 5. Save corrected outputs
# --------------------------------------------------

movies.to_csv(
    ANALYSIS_DIR / "movie_financial_metrics.csv",
    index=False
)

comparison_df.to_csv(
    ANALYSIS_DIR / "licensing_comparison.csv",
    index=False
)

for period_name, units in unit_datasets.items():

    filename = (
        "licensing_units_all_years.csv"
        if period_name == "All release years"
        else "licensing_units_2000_onward.csv"
    )

    units.to_csv(ANALYSIS_DIR / filename, index=False)

print("\nCorrected comparison outputs saved.")

# --------------------------------------------------
# 5. Save analysis outputs
# --------------------------------------------------

movies.to_csv(
    ANALYSIS_DIR / "movie_financial_metrics.csv",
    index=False
)

comparison_df.to_csv(
    ANALYSIS_DIR / "licensing_comparison.csv",
    index=False
)

for period_name, units in unit_datasets.items():
    file_name = (
        "licensing_units_all_years.csv"
        if period_name == "All release years"
        else "licensing_units_2000_onward.csv"
    )

    units.to_csv(
        ANALYSIS_DIR / file_name,
        index=False
    )

print("\nANALYSIS OUTPUTS SAVED")
print("movie_financial_metrics.csv")
print("licensing_comparison.csv")
print("licensing_units_all_years.csv")
print("licensing_units_2000_onward.csv")

# --------------------------------------------------
# 6. Basic integrity checks
# --------------------------------------------------

assert movies["tmdb_id"].is_unique
assert comparison_df["period"].nunique() == 2
assert set(comparison_df["licensing_type"]) == {
    "Franchise bundle",
    "Standalone film"
}

print("\nAll financial-analysis integrity checks passed.")


Project root: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis
Processed data directory: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis\data\processed

All release years
licensing_type
Franchise bundle                    87
Standalone film                     16
Single-film collection candidate     9
Name: count, dtype: int64

2000 onward
licensing_type
Franchise bundle                    56
Standalone film                      8
Single-film collection candidate     7
Name: count, dtype: int64

LICENSING PERFORMANCE COMPARISON
                                                  analysis            period   licensing_type  licensing_units  films  total_budget  total_revenue  gross_revenue_less_budget  weighted_gross_return_pct  median_unit_gross_return_pct  median_revenue_multiple
                          Primary: multi-film bundles only All release years Franchise bundle               23     87   10643250000    58768179240                48124929240      

In [ ]:
from src.analysis import analyze_talent


# --------------------------------------------------
# 6. Director and lead-actor performance analysis
# --------------------------------------------------

cast = pd.read_csv(ANALYSIS_DIR / "movie_cast.csv")
crew = pd.read_csv(ANALYSIS_DIR / "movie_crew.csv")

# Standardise IDs for reliable joins.
cast["tmdb_id"] = pd.to_numeric(cast["tmdb_id"], errors="coerce")
crew["tmdb_id"] = pd.to_numeric(crew["tmdb_id"], errors="coerce")

# Financial information for each movie.
movie_metrics = movies[
    [
        "tmdb_id", "title", "release_year",
        "budget", "revenue", "profit", "movie_roi_pct"
    ]
].copy()

movie_metrics["tmdb_id"] = pd.to_numeric(
    movie_metrics["tmdb_id"], errors="coerce"
)




# Identify directors.
directors = crew[
    crew["job"].astype("string").str.strip().str.casefold()
    == "director"
].copy()

# Identify the three highest-billed actors per movie.
cast["cast_order"] = pd.to_numeric(
    cast["cast_order"], errors="coerce"
)

lead_actors = cast[
    cast["cast_order"].between(0, 2)
].copy()


director_results = []
actor_results = []

for period_name in ["All release years", "2000 onward"]:

    director_all, director_ranked = analyze_talent(
        directors, "Director", period_name, movie_metrics
    )

    actor_all, actor_ranked = analyze_talent(
        lead_actors, "Lead actor (top 3 billed)", period_name, movie_metrics
    )

    director_results.append(director_all)
    actor_results.append(actor_all)

    print(f"\n{'=' * 70}")
    print(f"DIRECTOR PERFORMANCE: {period_name}")
    print("=" * 70)
    print(director_ranked.head(15).round(2).to_string(index=False))

    print(f"\n{'=' * 70}")
    print(f"LEAD ACTOR PERFORMANCE: {period_name}")
    print("=" * 70)
    print(actor_ranked.head(15).round(2).to_string(index=False))


# Combine and save complete performance tables.
director_performance = pd.concat(
    director_results, ignore_index=True
)

actor_performance = pd.concat(
    actor_results, ignore_index=True
)

director_performance.to_csv(
    ANALYSIS_DIR / "director_performance.csv",
    index=False
)

actor_performance.to_csv(
    ANALYSIS_DIR / "lead_actor_performance.csv",
    index=False
)

print("\nTALENT ANALYSIS OUTPUTS SAVED")
print("director_performance.csv")
print("lead_actor_performance.csv")



DIRECTOR PERFORMANCE: All release years
 person_id                 name  film_count                                                                                                                                                                                                                                                          titles  total_budget  total_revenue  gross_revenue_less_budget  median_movie_gross_return_pct  mean_movie_gross_return_pct  weighted_gross_return_pct     role            period
       488     Steven Spielberg           8                              Indiana Jones and the Kingdom of the Crystal Skull | Indiana Jones and the Last Crusade | Indiana Jones and the Temple of Doom | Jaws | Jurassic Park | Raiders of the Lost Ark | Schindler's List | The Lost World: Jurassic Park     444000000     4452952877                 4008952877                        1225.21                      1835.11                     902.92 Director All release years
        24      Rob

In [3]:
print(
    comparison_df[
        comparison_df["analysis"].str.startswith("Sensitivity")
    ].round(2).to_string(index=False)
)

                                                  analysis            period   licensing_type  licensing_units  films  total_budget  total_revenue  gross_revenue_less_budget  weighted_gross_return_pct  median_unit_gross_return_pct  median_revenue_multiple
Sensitivity: single-film collections treated as standalone All release years Franchise bundle               23     87   10643250000    58768179240                48124929240                     452.16                        455.11                     5.55
Sensitivity: single-film collections treated as standalone All release years  Standalone film               25     25    1502700000     8393602523                 6890902523                     458.57                        400.00                     5.00
Sensitivity: single-film collections treated as standalone       2000 onward Franchise bundle               17     56    8968000000    46195146097                37227146097                     415.11                        361.15  

In [4]:
print("\nSINGLE-FILM COLLECTION CANDIDATES")

for period_name, period_movies in periods.items():
    collection_sizes = (
        period_movies.dropna(subset=["collection_id"])
        .groupby("collection_id")["tmdb_id"]
        .nunique()
    )

    candidates = period_movies[
        period_movies["collection_id"].notna()
        & period_movies["collection_id"].map(collection_sizes).eq(1)
    ]

    print(f"\n{period_name}:")
    print(
        candidates[["title", "release_year", "collection_name"]]
        .to_string(index=False)
    )


SINGLE-FILM COLLECTION CANDIDATES

All release years:
                   title  release_year                    collection_name
            Blade Runner          1982            Blade Runner Collection
The Silence of the Lambs          1991     The Hannibal Lecter Collection
                    Jaws          1975                    Jaws Collection
          Monsters, Inc.          2001          Monsters, Inc. Collection
            Ghostbusters          1984            Ghostbusters Collection
           The Lion King          1994           The Lion King Collection
         The Incredibles          2004         The Incredibles Collection
              The Smurfs          2011 The Smurfs (Theatrical) Collection
      Mad Max: Fury Road          2015                 Mad Max Collection

2000 onward:
                                             title  release_year                    collection_name
Indiana Jones and the Kingdom of the Crystal Skull          2008           Indiana Jones Co

In [5]:
cast = pd.read_csv(ANALYSIS_DIR / "movie_cast.csv")
crew = pd.read_csv(ANALYSIS_DIR / "movie_crew.csv")

print("CAST COLUMNS")
print(cast.columns.tolist())
print(cast.head(3).to_string(index=False))

print("\nCREW COLUMNS")
print(crew.columns.tolist())
print(crew.head(3).to_string(index=False))

CAST COLUMNS
['tmdb_id', 'person_id', 'credit_id', 'name', 'character', 'cast_order', 'gender', 'known_for_department']
 tmdb_id  person_id                credit_id          name            character  cast_order  gender known_for_department
      11          2 52fe420dc3a36847f8000441   Mark Hamill       Luke Skywalker           0       2               Acting
      11          3 52fe420dc3a36847f8000445 Harrison Ford             Han Solo           1       2               Acting
      11          4 52fe420dc3a36847f8000449 Carrie Fisher Princess Leia Organa           2       1               Acting

CREW COLUMNS
['tmdb_id', 'person_id', 'credit_id', 'name', 'department', 'job', 'gender', 'known_for_department']
 tmdb_id  person_id                credit_id           name department                     job  gender known_for_department
      11       7753 52fe420dc3a36847f8000473 Gilbert Taylor     Camera Director of Photography       2               Camera
      11        491 52fe420dc3a36

In [7]:

# Inspect the talent results, especially the post-2000 period

director_results = pd.read_csv(
    ANALYSIS_DIR / "director_performance.csv"
)

actor_results = pd.read_csv(
    ANALYSIS_DIR / "lead_actor_performance.csv"
)

print("\nTOP DIRECTORS — 2000 ONWARD")
display(
    director_results[
        director_results["period"] == "2000 onward"
    ].sort_values(
        ["median_movie_gross_return_pct", "film_count"],
        ascending=[False, False]
    ).head(10)
)

print("\nTOP LEAD ACTORS — ALL RELEASE YEARS")
display(
    actor_results[
        actor_results["period"] == "All release years"
    ].sort_values(
        ["median_movie_gross_return_pct", "film_count"],
        ascending=[False, False]
    ).head(10)
)

print("\nTOP LEAD ACTORS — 2000 ONWARD")
display(
    actor_results[
        actor_results["period"] == "2000 onward"
    ].sort_values(
        ["median_movie_gross_return_pct", "film_count"],
        ascending=[False, False]
    ).head(10)
)



TOP DIRECTORS — 2000 ONWARD


,person_id,name,film_count,titles,total_budget,total_revenue,gross_revenue_less_budget,median_movie_gross_return_pct,mean_movie_gross_return_pct,weighted_gross_return_pct,role,period
73,608,Hayao Miyazaki,1,Spirited Away,19000000,274925095,255925095,1346.974184,1346.974184,1346.974184,Director,2000 onward
83,5714,Carlos Saldanha,2,Ice Age: Dawn of the Dinosaurs | Ice Age: The ...,170000000,1547685573,1377685573,805.728010,805.728010,810.403278,Director,2000 onward
91,10965,Chris Columbus,2,Harry Potter and the Chamber of Secrets | Harr...,225000000,1906063097,1681063097,750.094087,750.094087,747.139154,Director,2000 onward
96,12058,Vicky Jenson,1,Shrek,60000000,489676241,429676241,716.127068,716.127068,716.127068,Director,2000 onward
105,19271,Anthony Russo,2,Avengers: Endgame | Avengers: Infinity War,656000000,4995064817,4339064817,655.362681,655.362681,661.442807,Director,2000 onward
106,19272,Joe Russo,2,Avengers: Endgame | Avengers: Infinity War,656000000,4995064817,4339064817,655.362681,655.362681,661.442807,Director,2000 onward
81,5524,Andrew Adamson,2,Shrek | Shrek 2,210000000,1422218982,1212218982,618.911114,618.911114,577.247134,Director,2000 onward
69,108,Peter Jackson,6,The Hobbit: An Unexpected Journey | The Hobbit...,1016000000,5852068099,4836068099,572.698329,645.664616,475.990955,Director,2000 onward
82,5713,Chris Wedge,1,Ice Age,59000000,383257136,324257136,549.588366,549.588366,549.588366,Director,2000 onward
66,1,George Lucas,2,Star Wars: Episode II - Attack of the Clones |...,233000000,1499398328,1266398328,546.688831,546.688831,543.518596,Director,2000 onward



TOP LEAD ACTORS — ALL RELEASE YEARS


,person_id,name,film_count,titles,total_budget,total_revenue,gross_revenue_less_budget,median_movie_gross_return_pct,mean_movie_gross_return_pct,weighted_gross_return_pct,role,period
83,3037,Richard Dreyfuss,1,Jaws,7000000,470653000,463653000,6623.614286,6623.614286,6623.614286,Lead actor (top 3 billed),All release years
122,6355,Roy Scheider,1,Jaws,7000000,470653000,463653000,6623.614286,6623.614286,6623.614286,Lead actor (top 3 billed),All release years
137,8606,Robert Shaw,1,Jaws,7000000,470653000,463653000,6623.614286,6623.614286,6623.614286,Lead actor (top 3 billed),All release years
85,3084,Marlon Brando,1,The Godfather,6000000,245066411,239066411,3984.440183,3984.440183,3984.440183,Lead actor (top 3 billed),All release years
86,3085,James Caan,1,The Godfather,6000000,245066411,239066411,3984.440183,3984.440183,3984.440183,Lead actor (top 3 billed),All release years
19,190,Clint Eastwood,1,"The Good, the Bad and the Ugly",1200000,38900000,37700000,3141.666667,3141.666667,3141.666667,Lead actor (top 3 billed),All release years
92,3265,Eli Wallach,1,"The Good, the Bad and the Ugly",1200000,38900000,37700000,3141.666667,3141.666667,3141.666667,Lead actor (top 3 billed),All release years
101,4078,Lee Van Cleef,1,"The Good, the Bad and the Ugly",1200000,38900000,37700000,3141.666667,3141.666667,3141.666667,Lead actor (top 3 billed),All release years
0,2,Mark Hamill,3,Return of the Jedi | Star Wars | The Empire St...,61350000,1886498007,1825148007,2891.111111,3836.836159,2974.976377,Lead actor (top 3 billed),All release years
2,4,Carrie Fisher,3,Return of the Jedi | Star Wars | The Empire St...,61350000,1886498007,1825148007,2891.111111,3836.836159,2974.976377,Lead actor (top 3 billed),All release years



TOP LEAD ACTORS — 2000 ONWARD


,person_id,name,film_count,titles,total_budget,total_revenue,gross_revenue_less_budget,median_movie_gross_return_pct,mean_movie_gross_return_pct,weighted_gross_return_pct,role,period
283,19587,Rumi Hiiragi,1,Spirited Away,19000000,274925095,255925095,1346.974184,1346.974184,1346.974184,Lead actor (top 3 billed),2000 onward
284,19588,Miyu Irino,1,Spirited Away,19000000,274925095,255925095,1346.974184,1346.974184,1346.974184,Lead actor (top 3 billed),2000 onward
285,19589,Mari Natsuki,1,Spirited Away,19000000,274925095,255925095,1346.974184,1346.974184,1346.974184,Lead actor (top 3 billed),2000 onward
190,109,Elijah Wood,3,The Lord of the Rings: The Fellowship of the R...,266000000,2916544743,2650544743,1072.515696,999.926117,996.445392,Lead actor (top 3 billed),2000 onward
191,110,Viggo Mortensen,3,The Lord of the Rings: The Fellowship of the R...,266000000,2916544743,2650544743,1072.515696,999.926117,996.445392,Lead actor (top 3 billed),2000 onward
243,5723,John Leguizamo,3,Ice Age | Ice Age: Dawn of the Dinosaurs | Ice...,229000000,1930942709,1701942709,726.248445,720.348129,743.206423,Lead actor (top 3 billed),2000 onward
244,5724,Denis Leary,3,Ice Age | Ice Age: Dawn of the Dinosaurs | Ice...,229000000,1930942709,1701942709,726.248445,720.348129,743.206423,Lead actor (top 3 billed),2000 onward
276,15757,Ray Romano,3,Ice Age | Ice Age: Dawn of the Dinosaurs | Ice...,229000000,1930942709,1701942709,726.248445,720.348129,743.206423,Lead actor (top 3 billed),2000 onward
189,103,Mark Ruffalo,3,Avengers: Age of Ultron | Avengers: Endgame | ...,811000000,5866868987,5055868987,590.370689,605.000610,623.411712,Lead actor (top 3 billed),2000 onward
277,16828,Chris Evans,3,Avengers: Endgame | Avengers: Infinity War | T...,876000000,6513880332,5637880332,590.370689,633.698684,643.593645,Lead actor (top 3 billed),2000 onward


In [8]:

# Filter talent results to people with at least two films
# This avoids treating one-film results as repeat-performance evidence.

print("REPEAT DIRECTORS — ALL YEARS")
display(
    director_results[
        (director_results["period"] == "All release years") &
        (director_results["film_count"] >= 2)
    ].sort_values(
        ["median_movie_gross_return_pct", "film_count"],
        ascending=[False, False]
    ).head(10)
)

print("REPEAT DIRECTORS — 2000 ONWARD")
display(
    director_results[
        (director_results["period"] == "2000 onward") &
        (director_results["film_count"] >= 2)
    ].sort_values(
        ["median_movie_gross_return_pct", "film_count"],
        ascending=[False, False]
    ).head(10)
)

print("REPEAT LEAD ACTORS — ALL YEARS")
display(
    actor_results[
        (actor_results["period"] == "All release years") &
        (actor_results["film_count"] >= 2)
    ].sort_values(
        ["median_movie_gross_return_pct", "film_count"],
        ascending=[False, False]
    ).head(10)
)

print("REPEAT LEAD ACTORS — 2000 ONWARD")
display(
    actor_results[
        (actor_results["period"] == "2000 onward") &
        (actor_results["film_count"] >= 2)
    ].sort_values(
        ["median_movie_gross_return_pct", "film_count"],
        ascending=[False, False]
    ).head(10)
)


REPEAT DIRECTORS — ALL YEARS


,person_id,name,film_count,titles,total_budget,total_revenue,gross_revenue_less_budget,median_movie_gross_return_pct,mean_movie_gross_return_pct,weighted_gross_return_pct,role,period
6,488,Steven Spielberg,8,Indiana Jones and the Kingdom of the Crystal S...,444000000,4452952877,4008952877,1225.210701,1835.111805,902.917315,Director,All release years
3,24,Robert Zemeckis,4,Back to the Future | Back to the Future Part I...,154000000,1646873348,1492873348,931.883511,1084.308471,969.398278,Director,All release years
19,2710,James Cameron,3,Aliens | Terminator 2: Judgment Day | The Term...,126900000,779501830,652601830,890.899757,807.703571,514.264641,Director,All release years
31,7879,John Lasseter,2,Toy Story | Toy Story 2,120000000,898533373,778533373,844.916284,844.916284,648.777811,Director,All release years
26,5714,Carlos Saldanha,2,Ice Age: Dawn of the Dinosaurs | Ice Age: The ...,170000000,1547685573,1377685573,805.728010,805.728010,810.403278,Director,All release years
38,10965,Chris Columbus,2,Harry Potter and the Chamber of Secrets | Harr...,225000000,1906063097,1681063097,750.094087,750.094087,747.139154,Director,All release years
0,1,George Lucas,4,Star Wars | Star Wars: Episode I - The Phantom...,359000000,3321311744,2962311744,731.112894,2213.115963,825.156475,Director,All release years
16,1776,Francis Ford Coppola,3,The Godfather | The Godfather Part II | The Go...,73000000,484432473,411432473,689.230769,1608.980479,563.606127,Director,All release years
56,19271,Anthony Russo,2,Avengers: Endgame | Avengers: Infinity War,656000000,4995064817,4339064817,655.362681,655.362681,661.442807,Director,All release years
57,19272,Joe Russo,2,Avengers: Endgame | Avengers: Infinity War,656000000,4995064817,4339064817,655.362681,655.362681,661.442807,Director,All release years


REPEAT DIRECTORS — 2000 ONWARD


,person_id,name,film_count,titles,total_budget,total_revenue,gross_revenue_less_budget,median_movie_gross_return_pct,mean_movie_gross_return_pct,weighted_gross_return_pct,role,period
83,5714,Carlos Saldanha,2,Ice Age: Dawn of the Dinosaurs | Ice Age: The ...,170000000,1547685573,1377685573,805.728010,805.728010,810.403278,Director,2000 onward
91,10965,Chris Columbus,2,Harry Potter and the Chamber of Secrets | Harr...,225000000,1906063097,1681063097,750.094087,750.094087,747.139154,Director,2000 onward
105,19271,Anthony Russo,2,Avengers: Endgame | Avengers: Infinity War,656000000,4995064817,4339064817,655.362681,655.362681,661.442807,Director,2000 onward
106,19272,Joe Russo,2,Avengers: Endgame | Avengers: Infinity War,656000000,4995064817,4339064817,655.362681,655.362681,661.442807,Director,2000 onward
81,5524,Andrew Adamson,2,Shrek | Shrek 2,210000000,1422218982,1212218982,618.911114,618.911114,577.247134,Director,2000 onward
69,108,Peter Jackson,6,The Hobbit: An Unexpected Journey | The Hobbit...,1016000000,5852068099,4836068099,572.698329,645.664616,475.990955,Director,2000 onward
66,1,George Lucas,2,Star Wars: Episode II - Attack of the Clones |...,233000000,1499398328,1266398328,546.688831,546.688831,543.518596,Director,2000 onward
101,12891,Joss Whedon,2,Avengers: Age of Ultron | The Avengers,455000000,2924219209,2469219209,544.207407,544.207407,542.685540,Director,2000 onward
85,7087,Brad Bird,2,Mission: Impossible - Ghost Protocol | The Inc...,237000000,1326155472,1089155472,482.731388,482.731388,459.559271,Director,2000 onward
70,138,Quentin Tarantino,2,Kill Bill: Vol. 1 | Kill Bill: Vol. 2,60000000,333065537,273065537,455.109228,455.109228,455.109228,Director,2000 onward


REPEAT LEAD ACTORS — ALL YEARS


,person_id,name,film_count,titles,total_budget,total_revenue,gross_revenue_less_budget,median_movie_gross_return_pct,mean_movie_gross_return_pct,weighted_gross_return_pct,role,period
0,2,Mark Hamill,3,Return of the Jedi | Star Wars | The Empire St...,61350000,1886498007,1825148007,2891.111111,3836.836159,2974.976377,Lead actor (top 3 billed),All release years
2,4,Carrie Fisher,3,Return of the Jedi | Star Wars | The Empire St...,61350000,1886498007,1825148007,2891.111111,3836.836159,2974.976377,Lead actor (top 3 billed),All release years
73,2231,Samuel L. Jackson,2,Die Hard: With a Vengeance | Pulp Fiction,98000000,580030428,482030428,1440.444577,1440.444577,491.867784,Lead actor (top 3 billed),All release years
1,3,Harrison Ford,8,Blade Runner | Indiana Jones and the Kingdom o...,368350000,3912061512,3543711512,1379.996700,1991.063494,962.050091,Lead actor (top 3 billed),All release years
35,650,Karen Allen,2,Indiana Jones and the Kingdom of the Crystal S...,203000000,1176562004,973562004,1195.732030,1195.732030,479.587194,Lead actor (top 3 billed),All release years
110,4785,Jeff Goldblum,2,Jurassic Park | The Lost World: Jurassic Park,136000000,1677093229,1541093229,1163.768381,1163.768381,1133.156786,Lead actor (top 3 billed),All release years
10,109,Elijah Wood,3,The Lord of the Rings: The Fellowship of the R...,266000000,2916544743,2650544743,1072.515696,999.926117,996.445392,Lead actor (top 3 billed),All release years
11,110,Viggo Mortensen,3,The Lord of the Rings: The Fellowship of the R...,266000000,2916544743,2650544743,1072.515696,999.926117,996.445392,Lead actor (top 3 billed),All release years
79,2712,Michael Biehn,2,Aliens | The Terminator,24900000,261687655,236787655,1007.724878,1007.724878,950.954438,Lead actor (top 3 billed),All release years
163,16940,Jeremy Irons,2,Die Hard: With a Vengeance | The Lion King,135000000,1129557227,994557227,951.673771,951.673771,736.709057,Lead actor (top 3 billed),All release years


REPEAT LEAD ACTORS — 2000 ONWARD


,person_id,name,film_count,titles,total_budget,total_revenue,gross_revenue_less_budget,median_movie_gross_return_pct,mean_movie_gross_return_pct,weighted_gross_return_pct,role,period
190,109,Elijah Wood,3,The Lord of the Rings: The Fellowship of the R...,266000000,2916544743,2650544743,1072.515696,999.926117,996.445392,Lead actor (top 3 billed),2000 onward
191,110,Viggo Mortensen,3,The Lord of the Rings: The Fellowship of the R...,266000000,2916544743,2650544743,1072.515696,999.926117,996.445392,Lead actor (top 3 billed),2000 onward
243,5723,John Leguizamo,3,Ice Age | Ice Age: Dawn of the Dinosaurs | Ice...,229000000,1930942709,1701942709,726.248445,720.348129,743.206423,Lead actor (top 3 billed),2000 onward
244,5724,Denis Leary,3,Ice Age | Ice Age: Dawn of the Dinosaurs | Ice...,229000000,1930942709,1701942709,726.248445,720.348129,743.206423,Lead actor (top 3 billed),2000 onward
276,15757,Ray Romano,3,Ice Age | Ice Age: Dawn of the Dinosaurs | Ice...,229000000,1930942709,1701942709,726.248445,720.348129,743.206423,Lead actor (top 3 billed),2000 onward
189,103,Mark Ruffalo,3,Avengers: Age of Ultron | Avengers: Endgame | ...,811000000,5866868987,5055868987,590.370689,605.000610,623.411712,Lead actor (top 3 billed),2000 onward
277,16828,Chris Evans,3,Avengers: Endgame | Avengers: Infinity War | T...,876000000,6513880332,5637880332,590.370689,633.698684,643.593645,Lead actor (top 3 billed),2000 onward
220,1327,Ian McKellen,6,The Hobbit: An Unexpected Journey | The Hobbit...,1016000000,5852068099,4836068099,572.698329,645.664616,475.990955,Lead actor (top 3 billed),2000 onward
202,524,Natalie Portman,2,Star Wars: Episode II - Attack of the Clones |...,233000000,1499398328,1266398328,546.688831,546.688831,543.518596,Lead actor (top 3 billed),2000 onward
230,3061,Ewan McGregor,2,Star Wars: Episode II - Attack of the Clones |...,233000000,1499398328,1266398328,546.688831,546.688831,543.518596,Lead actor (top 3 billed),2000 onward


In [9]:

# Shortlist candidates from the primary and sensitivity analyses

print("TOP FRANCHISE BUNDLES — ALL YEARS")
display(
    unit_datasets["All release years"]
    .query("licensing_type == 'Franchise bundle'")
    .sort_values(
        ["gross_revenue_to_budget_pct", "total_revenue"],
        ascending=[False, False]
    )
    [["titles", "film_count", "total_budget", "total_revenue",
      "gross_revenue_less_budget", "gross_revenue_to_budget_pct"]]
    .head(10)
)

print("TOP FRANCHISE BUNDLES — 2000 ONWARD")
display(
    unit_datasets["2000 onward"]
    .query("licensing_type == 'Franchise bundle'")
    .sort_values(
        ["gross_revenue_to_budget_pct", "total_revenue"],
        ascending=[False, False]
    )
    [["titles", "film_count", "total_budget", "total_revenue",
      "gross_revenue_less_budget", "gross_revenue_to_budget_pct"]]
    .head(10)
)

print("STANDALONE LICENSING CANDIDATES — ALL YEARS")
display(
    sensitivity_datasets["All release years"]
    .query("licensing_type == 'Standalone film'")
    .sort_values(
        ["gross_revenue_to_budget_pct", "total_revenue"],
        ascending=[False, False]
    )
    [["titles", "film_count", "total_budget", "total_revenue",
      "gross_revenue_less_budget", "gross_revenue_to_budget_pct"]]
    .head(10)
)

print("STANDALONE LICENSING CANDIDATES — 2000 ONWARD")
display(
    sensitivity_datasets["2000 onward"]
    .query("licensing_type == 'Standalone film'")
    .sort_values(
        ["gross_revenue_to_budget_pct", "total_revenue"],
        ascending=[False, False]
    )
    [["titles", "film_count", "total_budget", "total_revenue",
      "gross_revenue_less_budget", "gross_revenue_to_budget_pct"]]
    .head(10)
)


TOP FRANCHISE BUNDLES — ALL YEARS


,titles,film_count,total_budget,total_revenue,gross_revenue_less_budget,gross_revenue_to_budget_pct
2,The Lord of the Rings: The Fellowship of the R...,3,266000000,2916544743,2650544743,996.445392
0,Star Wars | The Empire Strikes Back | Return o...,6,409350000,4432411744,4023061744,982.792658
11,Back to the Future | Back to the Future Part I...,3,99000000,969485632,870485632,879.278416
16,Jurassic Park | The Lost World: Jurassic Park ...,3,229000000,2045874038,1816874038,793.394776
19,Ice Age | Ice Age: The Meltdown | Ice Age: Daw...,3,229000000,1930942709,1701942709,743.206423
21,The Avengers | Avengers: Age of Ultron | Aveng...,4,1111000000,7919284026,6808284026,612.806843
20,Raiders of the Lost Ark | Indiana Jones and th...,4,279000000,1983841081,1704841081,611.054151
8,The Godfather | The Godfather Part II | The Go...,3,73000000,484432473,411432473,563.606127
1,Toy Story | Toy Story 2 | Toy Story 3 | Toy St...,4,495000000,3039690868,2544690868,514.078963
4,Harry Potter and the Philosopher's Stone | Har...,8,1280000000,7759777709,6479777709,506.232634


TOP FRANCHISE BUNDLES — 2000 ONWARD


,titles,film_count,total_budget,total_revenue,gross_revenue_less_budget,gross_revenue_to_budget_pct
2,The Lord of the Rings: The Fellowship of the R...,3,266000000,2916544743,2650544743,996.445392
14,Ice Age | Ice Age: The Meltdown | Ice Age: Daw...,3,229000000,1930942709,1701942709,743.206423
15,The Avengers | Avengers: Age of Ultron | Aveng...,4,1111000000,7919284026,6808284026,612.806843
0,Star Wars: Episode II - Attack of the Clones |...,2,233000000,1499398328,1266398328,543.518596
4,Harry Potter and the Philosopher's Stone | Har...,8,1280000000,7759777709,6479777709,506.232634
1,Toy Story 3 | Toy Story 4,2,375000000,2141157495,1766157495,470.975332
6,Shrek | Shrek 2 | Shrek the Third | Shrek Fore...,4,535000000,2983130036,2448130036,457.594399
9,Kill Bill: Vol. 1 | Kill Bill: Vol. 2,2,60000000,333065537,273065537,455.109228
12,The Bourne Identity | The Bourne Supremacy | T...,3,205000000,945358579,740358579,361.150526
5,Iron Man | Iron Man 2 | Iron Man 3,3,540000000,2424684758,1884684758,349.015696


STANDALONE LICENSING CANDIDATES — ALL YEARS


,titles,film_count,total_budget,total_revenue,gross_revenue_less_budget,gross_revenue_to_budget_pct
37,Jaws,1,7000000,470653000,463653000,6623.614286
35,"The Good, the Bad and the Ugly",1,1200000,38900000,37700000,3141.666667
41,Pulp Fiction,1,8000000,213928762,205928762,2574.109525
46,The Lion King,1,45000000,763455561,718455561,1596.567913
34,Schindler's List,1,22000000,321365567,299365567,1360.752577
26,Spirited Away,1,19000000,274925095,255925095,1346.974184
31,The Silence of the Lambs,1,19000000,272742922,253742922,1335.489063
27,Forrest Gump,1,55000000,677387716,622387716,1131.614029
39,Ghostbusters,1,30000000,296640120,266640120,888.800400
47,The Incredibles,1,92000000,631442092,539442092,586.350100


STANDALONE LICENSING CANDIDATES — 2000 ONWARD


,titles,film_count,total_budget,total_revenue,gross_revenue_less_budget,gross_revenue_to_budget_pct
19,Spirited Away,1,19000000,274925095,255925095,1346.974184
31,The Incredibles,1,92000000,631442092,539442092,586.350100
24,Inception,1,160000000,839030630,679030630,424.394144
26,The Smurfs,1,110000000,563749323,453749323,412.499385
27,"Monsters, Inc.",1,130000000,650000000,520000000,400.000000
22,Interstellar,1,165000000,746606706,581606706,352.488913
30,Memento,1,9000000,40060108,31060108,345.112311
23,Indiana Jones and the Kingdom of the Crystal S...,1,185000000,786636033,601636033,325.208666
25,Jurassic Park III,1,93000000,368780809,275780809,296.538504
21,Live Free or Die Hard,1,110000000,383531464,273531464,248.664967


In [10]:

# Export the key results into one readable text file

output_file = PROJECT_ROOT / "data" / "processed" / "final_analysis_results.txt"

with open(output_file, "w", encoding="utf-8") as f:

    sections = [
        ("TOP DIRECTORS — ALL YEARS",
         director_results[
             (director_results["period"] == "All release years") &
             (director_results["film_count"] >= 2)
         ]),
        ("TOP DIRECTORS — 2000 ONWARD",
         director_results[
             (director_results["period"] == "2000 onward") &
             (director_results["film_count"] >= 2)
         ]),
        ("TOP LEAD ACTORS — ALL YEARS",
         actor_results[
             (actor_results["period"] == "All release years") &
             (actor_results["film_count"] >= 2)
         ]),
        ("TOP LEAD ACTORS — 2000 ONWARD",
         actor_results[
             (actor_results["period"] == "2000 onward") &
             (actor_results["film_count"] >= 2)
         ]),
    ]

    for heading, df in sections:
        f.write("\n" + "=" * 75 + "\n")
        f.write(heading + "\n")
        f.write("=" * 75 + "\n")
        f.write(df.sort_values(
            ["median_movie_gross_return_pct", "film_count"],
            ascending=[False, False]
        ).head(10).to_string(index=False))
        f.write("\n")

    for period, dataset in unit_datasets.items():
        f.write("\n" + "=" * 75 + "\n")
        f.write(f"TOP FRANCHISE BUNDLES — {period}\n")
        f.write("=" * 75 + "\n")
        f.write(
            dataset.query("licensing_type == 'Franchise bundle'")
            .sort_values(
                ["gross_revenue_to_budget_pct", "total_revenue"],
                ascending=[False, False]
            ).head(10).to_string(index=False)
        )
        f.write("\n")

    for period, dataset in sensitivity_datasets.items():
        f.write("\n" + "=" * 75 + "\n")
        f.write(f"TOP STANDALONE LICENSING CANDIDATES — {period}\n")
        f.write("=" * 75 + "\n")
        f.write(
            dataset.query("licensing_type == 'Standalone film'")
            .sort_values(
                ["gross_revenue_to_budget_pct", "total_revenue"],
                ascending=[False, False]
            ).head(10).to_string(index=False)
        )
        f.write("\n")

print(f"Saved results to: {output_file}")


Saved results to: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis\data\processed\final_analysis_results.txt


In [11]:

print("STRICT STANDALONE FILMS — ALL YEARS")

display(
    unit_datasets["All release years"]
    .query("licensing_type == 'Standalone film'")
    .sort_values(
        ["gross_revenue_to_budget_pct", "total_revenue"],
        ascending=[False, False]
    )
    [["titles", "total_budget", "total_revenue",
      "gross_revenue_less_budget", "gross_revenue_to_budget_pct"]]
)

print("STRICT STANDALONE FILMS — 2000 ONWARD")

display(
    unit_datasets["2000 onward"]
    .query("licensing_type == 'Standalone film'")
    .sort_values(
        ["gross_revenue_to_budget_pct", "total_revenue"],
        ascending=[False, False]
    )
    [["titles", "total_budget", "total_revenue",
      "gross_revenue_less_budget", "gross_revenue_to_budget_pct"]]
)


STRICT STANDALONE FILMS — ALL YEARS


,titles,total_budget,total_revenue,gross_revenue_less_budget,gross_revenue_to_budget_pct
33,"The Good, the Bad and the Ugly",1200000,38900000,37700000,3141.666667
36,Pulp Fiction,8000000,213928762,205928762,2574.109525
32,Schindler's List,22000000,321365567,299365567,1360.752577
26,Spirited Away,19000000,274925095,255925095,1346.974184
27,Forrest Gump,55000000,677387716,622387716,1131.614029
30,Inception,160000000,839030630,679030630,424.394144
29,Interstellar,165000000,746606706,581606706,352.488913
38,Memento,9000000,40060108,31060108,345.112311
28,The Departed,90000000,291465000,201465000,223.850000
23,WALL·E,180000000,521311860,341311860,189.617700


STRICT STANDALONE FILMS — 2000 ONWARD


,titles,total_budget,total_revenue,gross_revenue_less_budget,gross_revenue_to_budget_pct
19,Spirited Away,19000000,274925095,255925095,1346.974184
22,Inception,160000000,839030630,679030630,424.394144
21,Interstellar,165000000,746606706,581606706,352.488913
24,Memento,9000000,40060108,31060108,345.112311
20,The Departed,90000000,291465000,201465000,223.850000
17,WALL·E,180000000,521311860,341311860,189.617700
18,The Prestige,40000000,109676311,69676311,174.190777
23,Requiem for a Dream,4500000,7390108,2890108,64.224622


In [12]:

# Create the client memo and decision notes

from pathlib import Path

reports_dir = PROJECT_ROOT / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

client_memo = r"""
# StreamVault Content Licensing Recommendation

**To:** StreamVault Decision Team  
**Subject:** Evidence-based shortlist for the $14 million content licensing decision

## Recommendation

Prioritise franchise bundles for further licensing due diligence, while retaining a smaller group of standalone films as alternatives. In the primary analysis, franchise bundles outperform standalone films on weighted gross revenue-to-budget return and median unit return in both the full historical sample and the post-2000 sample.

The recommended franchise shortlist is:

1. **The Lord of the Rings** — three films in the post-2000 sample; approximately $2.92 billion in combined reported revenue and a 996.45% gross revenue-to-budget return.
2. **Ice Age** — three films in the post-2000 sample; approximately $1.93 billion in combined reported revenue and a 743.21% return.
3. **The Avengers** — four films in the post-2000 sample; approximately $7.92 billion in combined reported revenue and a 612.81% return.

These are screening candidates, not confirmed available licensing packages. Their actual rights, territories, windows, exclusivity, costs and contractual restrictions must be verified before any commitment.

## Standalone film shortlist

The five strongest strict standalone candidates by gross revenue-to-budget return across all release years are:

| Film | Reported revenue | Gross return |
|---|---:|---:|
| The Good, the Bad and the Ugly | $38.90M | 3,141.67% |
| Pulp Fiction | $213.93M | 2,574.11% |
| Schindler's List | $321.37M | 1,360.75% |
| Spirited Away | $274.93M | 1,346.97% |
| Forrest Gump | $677.39M | 1,131.61% |

For the post-2000 period, the five-film shortlist changes to **Spirited Away, Inception, Interstellar, Memento and The Departed**.

## Does the recommendation change after 2000?

The broad recommendation does not change: franchise bundles remain stronger than standalone films in the primary comparison. The standalone shortlist does change substantially when restricted to films released from 2000 onward.

A sensitivity test that treats single-film collection entries as standalone licensing units slightly changes the all-years weighted comparison: standalone units reach 458.57%, compared with 452.16% for franchise bundles. However, franchise bundles retain a higher median unit return. In the post-2000 sensitivity analysis, franchises outperform standalone units on both weighted and median returns.

## Talent evidence

Repeat appearances by directors and lead actors provide additional context for the franchise candidates. Examples include Peter Jackson across the sampled Lord of the Rings films, Carlos Saldanha across Ice Age titles, and the Russo brothers across Avengers: Infinity War and Avengers: Endgame. These are descriptive associations and do not establish that talent caused financial performance.

## Decision and next steps

Use the shortlist to guide rights and commercial due diligence. Do not commit the $14 million solely on the basis of historical box-office performance. Before approval, obtain title-level licensing quotations and verify rights availability, territory, term, exclusivity, distribution restrictions, expected audience demand and projected contribution after all relevant costs.

**Conclusion:** franchise bundles are the stronger initial screening direction, with the five standalone titles retained as alternatives. The final purchase decision should depend on verified rights and deal economics, not box-office ratios alone.
"""

decision_notes = r"""
# StreamVault Licensing Analysis — Decision Notes

## 1. Decision question

Assess whether StreamVault should prioritise franchise bundles or standalone films when screening content opportunities for a proposed $14 million licensing decision.

The available dataset contains 112 successfully retrieved movie records. It is a screening dataset, not a complete catalogue of licensable rights.

## 2. Data quality and scope

- The original source file contained 127 data rows.
- One row was blank and two values were invalid identifiers.
- Duplicate source entries were reconciled.
- Five valid-looking TMDb lookups returned 404 responses and were recorded as failures rather than silently dropped.
- The analysis uses 112 unique successfully retrieved movie records.
- Budget and revenue were populated with positive values for the 112 records, but this does not independently verify their accuracy.

The original source file was preserved. Identifier reconciliation, source-row lineage and acquisition failures are retained in the project data.

## 3. Licensing-unit definitions

The primary classification treats a collection with at least two sampled films in the relevant analysis period as a franchise bundle. Films without a collection ID are classified as standalone films.

Films belonging to a collection with only one sampled film in the period are excluded from the primary comparison as single-film collection candidates.

A sensitivity analysis instead treats these single-film collection candidates as standalone licensing units. This is a modelling assumption, not confirmation that the film is independent intellectual property.

The classification is recalculated separately for all release years and for films released from 2000 onward.

## 4. Financial methodology

For each film:

- Gross revenue less production budget = reported revenue minus reported production budget.
- Gross revenue-to-budget return (%) = (reported revenue minus reported production budget) / reported production budget × 100.
- Revenue multiple = reported revenue / reported production budget.

For licensing units, budgets and revenues are aggregated across the films in each unit. Weighted gross return is calculated using the aggregated budget and revenue. Median unit return is also reported to reduce reliance on a single very large unit.

These metrics are box-office screening proxies. Gross revenue less production budget is not studio profit. The calculations exclude marketing, distribution, exhibitor shares, taxes, financing costs and other expenses. They also do not measure licensing fees or StreamVault's expected cash flows.

## 5. Main findings

In the primary comparison, franchise bundles outperform standalone films on weighted and median gross return in both periods.

For all release years:
- Franchise bundles: 452.16% weighted gross return; 455.11% median unit return.
- Standalone films: 384.95% weighted gross return; 284.48% median unit return.

For films released from 2000 onward:
- Franchise bundles: 415.11% weighted gross return; 361.15% median unit return.
- Standalone films: 324.04% weighted gross return; 284.48% median unit return.

In the sensitivity analysis, single-film collection candidates are treated as standalone licensing units:
- All years: standalone units have a 458.57% weighted return versus 452.16% for franchises, but the franchise median remains higher (455.11% versus 400%).
- Post-2000: franchises outperform standalone units on weighted return (415.11% versus 328.84%) and median return (361.15% versus 325.21%).

The sensitivity result shows that classification assumptions can affect the overall comparison. The primary result should therefore be read alongside the sensitivity result.

## 6. Shortlist

### Franchise screening candidates
1. The Lord of the Rings
2. Ice Age
3. The Avengers

The post-2000 franchise shortlist balances relative returns and total reported revenue. Harry Potter is a further alternative worth considering.

### Strict standalone candidates — all years
1. The Good, the Bad and the Ugly
2. Pulp Fiction
3. Schindler's List
4. Spirited Away
5. Forrest Gump

### Strict standalone candidates — post-2000
1. Spirited Away
2. Inception
3. Interstellar
4. Memento
5. The Departed

The standalone shortlist changes materially when the analysis is restricted to post-2000 releases.

## 7. Talent analysis

Director and lead-actor performance is assessed by joining credits to movie financial metrics. Lead actors are defined as the first three cast members by cast order. Directors are identified using the crew job value "Director".

The analysis considers repeat contributors with at least two sampled films when discussing consistency. One-film observations are not sufficient evidence of repeat performance.

Examples of recurring contributors in the post-2000 sample include Peter Jackson, Carlos Saldanha, Anthony Russo and Joe Russo, as well as Elijah Wood, Viggo Mortensen, John Leguizamo, Denis Leary and Ray Romano.

These associations are descriptive only. They do not demonstrate that an individual director or actor caused a film's financial result.

## 8. Assumptions, limitations and risks

1. **Third-party financial data:** TMDb budget and revenue fields may be incomplete, estimated or inconsistent.
2. **No inflation adjustment:** nominal figures across different decades are not directly comparable in purchasing-power terms.
3. **Box office is not licensing value:** historical theatrical revenue does not establish future streaming demand, subscription value or licensing affordability.
4. **No licensing quotes:** the dataset does not contain title-level licensing fees, rights availability or contractual terms.
5. **Collection membership:** TMDb collection membership is a proxy and does not establish ownership, rights availability or a package that can be licensed.
6. **Sample bias:** the 112 successful records are a limited subset of the source list and may not represent the wider content market.
7. **Talent attribution:** financial outcomes cannot be attributed to individual talent from these descriptive comparisons.
8. **Unit aggregation:** summing revenue and budget across films can favour larger bundles and does not account for release timing or differences in title-level performance.

## 9. Decision conditions

Before committing funds, StreamVault should:
- Confirm ownership and licensable rights for each title.
- Verify territory, term, exclusivity, platform, language and window restrictions.
- Obtain actual licensing quotations and payment schedules.
- Forecast audience demand and expected incremental value to StreamVault.
- Calculate contribution after licensing fees and all relevant operating and distribution costs.
- Compare the proposed package against alternative content opportunities.
- Conduct legal and commercial diligence before signing.

The $14 million decision cannot be justified by gross revenue-to-budget ratios alone. The analysis supports prioritising candidates for diligence, not approving a purchase price.

## 10. Recommended improvement

For a catalogue expected to grow to approximately 500 titles and be refreshed quarterly, implement an incremental, resumable acquisition and analysis pipeline.

The existing acquisition process already uses caching, request throttling, transient retries, failure logging, identifier reconciliation and source-row lineage. Retain these controls and extend the pipeline with:
- Scheduled quarterly refreshes.
- Explicit schema and data-quality validation.
- Change detection for financial and metadata fields.
- Automated unit tests for classification and financial calculations.
- A dated snapshot of each quarterly output.
- A run summary reporting processed, cached, updated and failed records.

Measure the improvement using runtime, API request volume, cache-hit rate, failure rate, duplicate rate and the proportion of records reconciled to source rows. Compare these measures across a repeat run and a simulated larger catalogue. Do not claim a measured performance gain until the benchmark has actually been run and recorded.
"""

(reports_dir / "client_memo.md").write_text(
    client_memo.strip() + "\n", encoding="utf-8"
)

(reports_dir / "decision_notes.md").write_text(
    decision_notes.strip() + "\n", encoding="utf-8"
)

print("Created:")
print(reports_dir / "client_memo.md")
print(reports_dir / "decision_notes.md")


Created:
c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis\reports\client_memo.md
c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis\reports\decision_notes.md


In [13]:

# Add final analysis summary to README.md without replacing existing content

readme_path = PROJECT_ROOT / "README.md"

final_readme_section = """

## Final Licensing Analysis

### Objective
Evaluate franchise bundles against standalone films to identify content candidates for further due diligence in relation to the proposed $14 million licensing decision.

### Dataset and coverage
- 112 successfully retrieved, unique movie records.
- Source-row reconciliation and acquisition failures documented.
- Financial comparisons conducted for all release years and separately for films released from 2000 onward.
- Director and lead-actor performance explored using TMDb credits.

### Key financial findings

| Period | Licensing type | Weighted gross return | Median unit return |
|---|---|---:|---:|
| All years | Franchise bundles | 452.16% | 455.11% |
| All years | Standalone films | 384.95% | 284.48% |
| 2000 onward | Franchise bundles | 415.11% | 361.15% |
| 2000 onward | Standalone films | 324.04% | 284.48% |

The primary comparison favours franchise bundles in both periods. A sensitivity analysis treating single-film collection candidates as standalone licensing units slightly changes the all-years weighted comparison, so the classification assumption is documented in the decision notes.

### Recommended screening shortlist

**Franchise bundles**
1. The Lord of the Rings
2. Ice Age
3. The Avengers

**Strict standalone films — all years**
1. The Good, the Bad and the Ugly
2. Pulp Fiction
3. Schindler's List
4. Spirited Away
5. Forrest Gump

**Strict standalone films — 2000 onward**
1. Spirited Away
2. Inception
3. Interstellar
4. Memento
5. The Departed

### Main limitations
- Financial figures are third-party reported nominal values and have not been adjusted for inflation.
- Gross revenue less production budget is not studio profit or licensing return.
- Collection membership does not prove that rights are available as a package.
- The analysis does not contain actual licensing quotations, rights terms or StreamVault-specific demand forecasts.
- Talent results are descriptive associations, not causal evidence.

The shortlist is intended to guide further rights and commercial due diligence. It is not, by itself, approval to spend the proposed licensing budget.

### Final deliverables
- `notebooks/03_streamvault_analysis.ipynb` — reproducible financial, sensitivity and talent analysis.
- `data/processed/licensing_comparison.csv` — comparison results.
- `data/processed/licensing_units_all_years.csv` — all-years licensing units.
- `data/processed/licensing_units_2000_onward.csv` — post-2000 licensing units.
- `data/processed/movie_financial_metrics.csv` — film-level financial metrics.
- `data/processed/director_performance.csv` — director performance results.
- `data/processed/lead_actor_performance.csv` — lead-actor performance results.
- `reports/client_memo.md` — client-facing recommendation.
- `reports/decision_notes.md` — methodology, assumptions, limitations and improvement plan.
"""

marker = "## Final Licensing Analysis"

readme_text = readme_path.read_text(encoding="utf-8")

if marker not in readme_text:
    with readme_path.open("a", encoding="utf-8") as f:
        f.write(final_readme_section)
    print("README final analysis section appended.")
else:
    print("README already contains the final analysis section; no duplicate added.")

print("README path:", readme_path)


README final analysis section appended.
README path: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis\README.md
